In [1]:
import pandas as pd
import numpy as np

In [2]:
train = pd.read_csv("../data/raw/train.csv")

In [3]:
import sys
from pathlib import Path

# Añadir src al path si el notebook está en la carpeta notebooks/
sys.path.append(str(Path("../src").resolve()))

from preprocessing import prep_train_df, prep_stores_df

In [4]:
import pandas as pd

train = pd.read_csv("../data/raw/train.csv")

df_train = prep_train_df(train, lags=[15, 30, 365])

In [5]:
df_train.columns

Index(['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion', 'weekday',
       'month', 'week', 'year', 'sales_lag15', 'sales_lag15_log',
       'sales_lag30', 'sales_lag30_log', 'sales_lag365', 'sales_lag365_log'],
      dtype='str')

In [6]:
df_train.head()

,id,date,store_nbr,family,sales,onpromotion,weekday,month,week,year,sales_lag15,sales_lag15_log,sales_lag30,sales_lag30_log,sales_lag365,sales_lag365_log
0,650430,2014-01-02,1,AUTOMOTIVE,3.0,0,3,1,1,2014,3.0,1.386294,3.0,1.386294,0.0,0.000000
1,652212,2014-01-03,1,AUTOMOTIVE,0.0,0,4,1,1,2014,1.0,0.693147,2.0,1.098612,2.0,1.098612
2,653994,2014-01-04,1,AUTOMOTIVE,0.0,0,5,1,1,2014,0.0,0.000000,1.0,0.693147,3.0,1.386294
3,655776,2014-01-05,1,AUTOMOTIVE,1.0,0,6,1,1,2014,5.0,1.791759,0.0,0.000000,3.0,1.386294
4,657558,2014-01-06,1,AUTOMOTIVE,6.0,0,0,1,2,2014,5.0,1.791759,2.0,1.098612,5.0,1.791759


In [7]:
def prep_joins(df_sales: pd.DataFrame, df_stores: pd.DataFrame) -> pd.DataFrame:
    """Joins all the datasets"""

    df = df_sales.merge(df_stores, how="left", on=["store_nbr"])
    return df

In [8]:
stores = pd.read_csv("../data/raw/stores.csv")
df_stores = prep_stores_df(stores)

In [9]:
df_stores.columns

Index(['store_nbr', 'city', 'state', 'type', 'cluster'], dtype='str')

In [10]:
df_join = prep_joins(df_sales=df_train, df_stores=df_stores)

In [11]:
df_join.columns

Index(['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion', 'weekday',
       'month', 'week', 'year', 'sales_lag15', 'sales_lag15_log',
       'sales_lag30', 'sales_lag30_log', 'sales_lag365', 'sales_lag365_log',
       'city', 'state', 'type', 'cluster'],
      dtype='str')

In [34]:
holidays_events = pd.read_csv("../data/raw/holidays_events.csv")

In [ ]:
def prep_holidays_df(holidays_events: pd.DataFrame) -> pd.DataFrame:
    """Preprocess the holiday data set"""

    holidays_events = holidays_events[holidays_events["transferred"] == False].rename(
        columns={"type": "holiday_type"}
    )
    holidays_events["date"] = pd.to_datetime(holidays_events["date"])
    return holidays_events


def prep_joins(df_sales: pd.DataFrame, df_stores: pd.DataFrame) -> pd.DataFrame:
    """Joins all the datasets"""

    # Joins the sales history dataset with the stores dataset
    df = df_sales.merge(df_stores, how="left", on=["store_nbr"])

    # Joins with the holidays dataset
    city_holidays = holidays_events[holidays_events["locale"] == "Local"]
    national_holidays = holidays_events[holidays_events["locale"] == "National"]
    regional_holidays = holidays_events[holidays_events["locale"] == "Regional"]

    df = df.merge(
        city_holidays[["date", "locale_name"]],
        how="left",
        left_on=["date", "city"],
        right_on=["date", "locale_name"],
    )
    df = df.merge(
        regional_holidays[["date", "locale_name"]],
        how="left",
        left_on=["date", "state"],
        right_on=["date", "locale_name"],
    )
    df = df.merge(
        national_holidays[["date", "locale_name"]],
        how="left",
        left_on=["date"],
        right_on=["date"],
    )

    holiday_cols = ["locale_name_x", "locale_name_y", "locale_name"]
    df["is_holiday"] = df[holiday_cols].notna().any(axis=1).astype(int)

    df = df.drop(columns=holiday_cols)
    return df

In [36]:
df_join_h[df_join_h["date"] == "01-01-2015"]

,id,date,store_nbr,family,sales,onpromotion,weekday,month,week,year,...,sales_lag15_log,sales_lag30,sales_lag30_log,sales_lag365,sales_lag365_log,city,state,type,cluster,is_holiday
364,1297296,2015-01-01,1,AUTOMOTIVE,0.0,0,3,1,1,2015,...,1.791759,8.000,2.197225,2.000,1.098612,Quito,Pichincha,D,13,1
1687,1297297,2015-01-01,1,BABY CARE,0.0,0,3,1,1,2015,...,0.000000,0.000,0.000000,0.000,0.000000,Quito,Pichincha,D,13,1
3010,1297298,2015-01-01,1,BEAUTY,0.0,0,3,1,1,2015,...,1.609438,3.000,1.386294,1.000,0.693147,Quito,Pichincha,D,13,1
4333,1297299,2015-01-01,1,BEVERAGES,0.0,0,3,1,1,2015,...,7.775696,2254.000,7.720905,840.000,6.734592,Quito,Pichincha,D,13,1
5656,1297300,2015-01-01,1,BOOKS,0.0,0,3,1,1,2015,...,0.000000,0.000,0.000000,0.000,0.000000,Quito,Pichincha,D,13,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2351599,1298941,2015-01-01,54,POULTRY,0.0,0,3,1,1,2015,...,3.586874,57.891,4.075688,192.485,5.265200,El Carmen,Manabi,C,3,1
2352922,1298942,2015-01-01,54,PREPARED FOODS,0.0,0,3,1,1,2015,...,4.644391,104.000,4.653960,9.000,2.302585,El Carmen,Manabi,C,3,1
2354245,1298943,2015-01-01,54,PRODUCE,0.0,0,3,1,1,2015,...,6.446742,446.141,6.102874,0.000,0.000000,El Carmen,Manabi,C,3,1
2355568,1298944,2015-01-01,54,SCHOOL AND OFFICE SUPPLIES,0.0,0,3,1,1,2015,...,0.000000,0.000,0.000000,0.000,0.000000,El Carmen,Manabi,C,3,1
